# Hotel Recommendation System using Collaborative Filtering

## End-to-End MLOps Capstone Project

### Project Summary

The objective of this project is to develop a hotel recommendation system using collaborative filtering techniques. The system recommends hotels to users based on historical booking patterns and user-item interactions.

Item-to-item collaborative filtering with cosine similarity is used to identify similar hotels. The recommendation engine is integrated into the complete MLOps pipeline for deployment through the Flask API and Streamlit application.

## Problem Statement

The objective of this project is to recommend hotels that users are likely to prefer based on historical booking behavior.

The project includes:

- Data preprocessing
- User-Item Interaction Matrix
- Item-to-Item Collaborative Filtering
- Cosine Similarity
- Recommendation Evaluation
- Model Serialization

Github : https://github.com/Yogesh-46/My_Voyage_Analytics.git

In [1]:
import os
import joblib
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity

In [2]:
DATA_FILE = "/content/hotels.csv"

df_hotels = pd.read_csv(DATA_FILE)

df_hotels.head()

,travelCode,userCode,name,place,days,price,total,date
0,0,0,Hotel A,Florianopolis (SC),4,313.02,1252.08,09/26/2019
1,2,0,Hotel K,Salvador (BH),2,263.41,526.82,10/10/2019
2,7,0,Hotel K,Salvador (BH),3,263.41,790.23,11/14/2019
3,11,0,Hotel K,Salvador (BH),4,263.41,1053.64,12/12/2019
4,13,0,Hotel A,Florianopolis (SC),1,313.02,313.02,12/26/2019


In [3]:
print("Dataset Shape :", df_hotels.shape)

df_hotels.info()

df_hotels.describe()

Dataset Shape : (40552, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40552 entries, 0 to 40551
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   travelCode  40552 non-null  int64  
 1   userCode    40552 non-null  int64  
 2   name        40552 non-null  object 
 3   place       40552 non-null  object 
 4   days        40552 non-null  int64  
 5   price       40552 non-null  float64
 6   total       40552 non-null  float64
 7   date        40552 non-null  object 
dtypes: float64(2), int64(3), object(3)
memory usage: 2.5+ MB


,travelCode,userCode,days,price,total
count,40552.000000,40552.000000,40552.000000,40552.000000,40552.000000
mean,67911.794461,666.963726,2.499679,214.439554,536.229513
std,39408.199333,391.136794,1.119326,76.742305,319.331482
min,0.000000,0.000000,1.000000,60.390000,60.390000
25%,33696.750000,323.000000,1.000000,165.990000,247.620000
50%,67831.000000,658.000000,2.000000,242.880000,495.240000
75%,102211.250000,1013.000000,4.000000,263.410000,742.860000
max,135942.000000,1339.000000,4.000000,313.020000,1252.080000


In [4]:
df_hotels.isnull().sum()

,0
travelCode,0
userCode,0
name,0
place,0
days,0
price,0
total,0
date,0


In [5]:
user_item_matrix = df_hotels.groupby(
    [
        "userCode",
        "name"
    ]
).size().unstack(fill_value=0)

user_item_matrix.head()

name,Hotel A,Hotel AF,Hotel AU,Hotel BD,Hotel BP,Hotel BW,Hotel CB,Hotel K,Hotel Z
userCode,,,,,,,,,
0,3,4,2,4,1,2,1,7,3
1,0,1,0,0,1,0,0,0,0
2,6,2,3,2,2,7,3,5,6
3,10,6,7,11,2,9,7,6,2
4,7,6,7,5,3,6,11,7,4


In [6]:
item_similarity = cosine_similarity(
    user_item_matrix.T
)

item_similarity_df = pd.DataFrame(
    item_similarity,
    index=user_item_matrix.columns,
    columns=user_item_matrix.columns
)

item_similarity_df.head()

name,Hotel A,Hotel AF,Hotel AU,Hotel BD,Hotel BP,Hotel BW,Hotel CB,Hotel K,Hotel Z
name,,,,,,,,,
Hotel A,1.000000,0.637485,0.615908,0.647272,0.601950,0.575352,0.665928,0.658882,0.567153
Hotel AF,0.637485,1.000000,0.813391,0.832255,0.786290,0.737742,0.822187,0.832258,0.822291
Hotel AU,0.615908,0.813391,1.000000,0.803205,0.818174,0.723447,0.808265,0.814410,0.818708
Hotel BD,0.647272,0.832255,0.803205,1.000000,0.802433,0.764057,0.826321,0.826903,0.822003
Hotel BP,0.601950,0.786290,0.818174,0.802433,1.000000,0.729258,0.809667,0.799159,0.806633


In [7]:
hits = 0

total = 0

In [8]:
for user_id in user_item_matrix.index[:300]:

    user_vector = user_item_matrix.loc[user_id].copy()

    interacted_items = user_vector[user_vector > 0].index.tolist()

    if len(interacted_items) < 2:
        continue

    test_item = np.random.choice(interacted_items)

    user_vector[test_item] = 0

    predicted_scores = (
        user_vector.values
        @ item_similarity_df.values
    )

    scores_series = pd.Series(
        predicted_scores,
        index=user_item_matrix.columns
    )

    scores_series = scores_series.drop(
        index=user_vector[user_vector > 0].index
    )

    top_3 = scores_series.nlargest(3).index.tolist()

    if test_item in top_3:
        hits += 1

    total += 1

In [9]:
hit_rate = (hits/total)*100

print("Hit Rate @3 :", hit_rate)

Hit Rate @3 : 92.9530201342282


In [10]:
artifacts = {

    "similarity_matrix":item_similarity_df,

    "user_item_matrix":user_item_matrix,

    "raw_hotels_data":

    df_hotels[
        [
            "name",
            "place"
        ]
    ].drop_duplicates().set_index(
        "name"
    )["place"].to_dict()

}

joblib.dump(
    artifacts,
    "/content/recommender_artifacts.pkl"
)

['/content/recommender_artifacts.pkl']

In [11]:
artifacts = {

    "similarity_matrix":item_similarity_df,

    "user_item_matrix":user_item_matrix,

    "raw_hotels_data":

    df_hotels[
        [
            "name",
            "place"
        ]
    ].drop_duplicates().set_index(
        "name"
    )["place"].to_dict()

}

joblib.dump(
    artifacts,
    "/content/recommender_artifacts.pkl"
)

['/content/recommender_artifacts.pkl']

## Conclusion

A collaborative filtering recommendation system was successfully developed using item-to-item cosine similarity. A user-item interaction matrix was constructed from historical booking records, and hotel similarities were computed to generate personalized recommendations.

The recommendation engine was evaluated using Hit Rate @3 and serialized for deployment within the complete MLOps pipeline. The trained recommendation artifacts are integrated into the Flask API and Streamlit application for real-time hotel recommendations.